# Benchmark: `allc → mcds → h5ad` (ALLCools / pym3c) vs `cz → h5ad` (cytozip)

Compare the two routes from single-cell methylation calls to an `AnnData` (`.h5ad`) feature matrix
on the mouse developing frontal cortex snm3C-seq data set (10 age groups, ~6k cells each, P56 ~21.6k cells):

| | Pipeline A — **ALLCools / pym3c** | Pipeline B — **cytozip** |
|---|---|---|
| input | `mapped/<age>/allc/*.allc.tsv.gz` (tabix) | `mapped/<age>/cz/*.cz` (from `czip allc2cz`, reference = `mm10_with_chrL.allc.cz`) |
| step 1 | `allcools generate-dataset` → `<age>.100kb.mcds` (`count CGN,CHN`) | `czip cz_to_anndata -f 100000 -s mm10_with_chrL.CGN.cz` → `100kb-CGN.h5ad` |
| step 2 | `allcools generate-dataset` → `<age>.gene.mcds` (`count CGN,CHN`) | `czip cz_to_anndata -f 100000 -s mm10_with_chrL.CHN.cz` → `100kb-CHN.h5ad` |
| step 3 | `pym3c adata generate_region_adata` → `100kb-CGN.h5ad`, `100kb-CHN.h5ad` | `czip cz_to_anndata -f gene_2000_2000.bed.gz -s mm10_with_chrL.CGN.cz` → `gene-CGN.h5ad` |
| step 4/5 | `pym3c adata generate_gadata` ×2 → `gene-CGN.h5ad`, `gene-CHN.h5ad` | `czip cz_to_anndata -f gene_2000_2000.bed.gz -s mm10_with_chrL.CHN.cz` → `gene-CHN.h5ad` |
| output | 4 × h5ad, `.X` = mC fraction | 4 × h5ad, `.X` = mC fraction, `.layers['mc'|'cov']` raw counts |

Both pipelines get **identical resources** (one exclusive SLURM job per (pipeline, sample), `NCPU` cores,
`MEM_GB` GB) and are driven by the same commands pym3c uses in production
(`pym3c/notebooks/projects/mouse_pfc.md`). Every stage is wrapped by `scripts/bench_run.py`, which samples the
whole process tree once per second and records:

* `wall_s` — wall-clock time; `cpu_user_s` / `cpu_sys_s` — exact CPU time from `wait4()` (all workers included)
* `max_rss_sum_mb` — peak Σ RSS over all processes (what `top` / SLURM `MaxRSS`-like accounting shows)
* `max_pss_sum_mb` — peak Σ PSS (proportional set size: pages shared between fork workers counted once → *true* physical footprint)
* `max_rss_single_mb` — largest single-process RSS (what `/usr/bin/time %M` reports)
* `output_size_mb` — size of the produced file(s)

Everything lives in `~/Projects/test_cytozip/to_adata/` (scripts, logs, mcds, h5ad, tables, figures).
The one-off `allc → cz` conversion (`czip allc2cz`) is timed as well so it can optionally be charged to pipeline B.


## 0. How to run (SLURM)

Every step is one script under `scripts/`; running it **submits one SLURM job per sample** through
`~/MySnakemake/pbsgen/pbsgen_slurm.py` (prefix with `DRYRUN=1` to only print the commands, pass sample
names to restrict, `--run ...` is the worker mode executed inside the job).

```shell
cd ~/Projects/test_cytozip/to_adata
# (edit env.sh to change NCPU / MEM_GB / A2C_NCPU / SAMPLES if needed)

# 0. one-off preparation, interactive node (fast): check the pre-built CGN/CHN indexes
#    (~/Ref/mm10/mm10_with_chrL.{CGN,CHN}.cz), write tables/<age>.allc_path.tsv,
#    create mapped/<age>/cz, tables/n_cells.tsv
bash scripts/00_prepare.sh

# 1. allc -> cz, one job per age (96 CPUs / 200 GB), writes mapped/<age>/cz/<cell>.cz + timing / size logs
bash scripts/01_allc2cz.sh                 # all ages        (DRYRUN=1 ... to preview; `... E15 P0` for a subset)

# 2. pipeline A  allc -> mcds -> h5ad   (one job per age, 32 CPUs / 128 GB) — only needs the allc files
bash scripts/02_allcools_pipeline.sh

# 3. pipeline B  cz -> h5ad             (one job per age, 32 CPUs / 128 GB) — needs step 1 done for that age
bash scripts/03_cytozip_pipeline.sh

# 4. cell-number scaling series (nested random subsets of P56: 500 ... 16000 cells), after 01 finished for P56
bash scripts/04_make_subsets.sh            # interactive: tables/P56_sub*.allc_path.tsv + cz_subsets/P56_sub*/
bash scripts/05_scaling.sh                 # submits allcools + cytozip jobs for every subset

# 5. run the cells below to collect logs/*.tsv and build tables/ and figures/
```

Safety guards (override with `FORCE=1`): `01_allc2cz.sh` refuses to run when `mapped/<age>/cz/` already holds `.cz`
files (`czip allc2cz` skips existing outputs, so a rerun would record only the remaining work);
`03_cytozip_pipeline.sh` only submits an age whose `logs/allc2cz.<age>.tsv` has an `exit_code == 0` row.

Resulting layout:

```
to_adata/
├── env.sh                      shared config (paths, NCPU, MEM_GB, A2C_NCPU, samples) + submit_job()
├── scripts/
│   ├── bench_run.py            process-tree time / memory monitor  -> logs/<pipeline>.<name>.tsv
│   ├── 00_prepare.sh           check indexes, allc tables, cz dirs          (run directly)
│   ├── 01_allc2cz.sh           [samples...] | --run <age>                    -> mapped/<age>/cz/
│   ├── 02_allcools_pipeline.sh [samples...] | --run <name> <allc_table>     -> mcds/, allcools/<name>/*.h5ad
│   ├── 03_cytozip_pipeline.sh  [samples...] | --run <name> <cz_dir>         -> cytozip/<name>/*.h5ad
│   ├── 04_make_subsets.sh      P56 subsets                                   (run directly)
│   └── 05_scaling.sh           [sizes...]  submits 02/03 workers for every subset
├── logs/  tables/  figures/  mcds/  allcools/  cytozip/  cz_subsets/  pbs/
```


In [ ]:
import os, sys, glob, re, textwrap, subprocess
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

WORKDIR = os.path.expanduser("~/Projects/test_cytozip/to_adata")
os.chdir(WORKDIR)
LOG_DIR, TABLE_DIR, FIG_DIR = "logs", "tables", "figures"
os.makedirs(TABLE_DIR, exist_ok=True); os.makedirs(FIG_DIR, exist_ok=True)

SAMPLES = ['E15', 'E17', 'P0', 'P3', 'P6', 'P9', 'P12', 'P13', 'P28', 'P56']
STAGES = ['100kb-CGN', '100kb-CHN', 'gene-CGN', 'gene-CHN']

# manuscript plotting style (same as notebooks/benchmark/benchmark_methylation_calling.ipynb)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'Arial'
mpl.rcParams['font.size'] = 8
mpl.rcParams['axes.linewidth'] = 0.6
COLORS = {'ALLCools': '#1f77b4', 'cytozip': '#d7263d', 'cytozip (+allc2cz)': '#f4a261'}
LABEL = {'allcools': 'ALLCools', 'cytozip': 'cytozip'}

def savefig(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(f"{FIG_DIR}/{name}.{ext}", bbox_inches='tight', dpi=300)
    print('saved', f"{FIG_DIR}/{name}.pdf/.png")

def despine(ax):
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)

print(subprocess.run(['bash', '-c', 'source env.sh; echo NCPU=$NCPU MEM_GB=$MEM_GB; czip --version 2>/dev/null || python -c "import cytozip;print(cytozip.__version__)"; allcools --version'],
                     capture_output=True, text=True).stdout)

### Scripts used (for the record)

In [ ]:
for f in ['env.sh', 'scripts/bench_run.py', 'scripts/00_prepare.sh', 'scripts/01_allc2cz.sh',
          'scripts/02_allcools_pipeline.sh', 'scripts/03_cytozip_pipeline.sh', 'scripts/04_make_subsets.sh', 'scripts/05_scaling.sh']:
    print('=' * 100); print('#', f); print('=' * 100)
    print(open(f).read())

## 1. Progress: which outputs exist?

In [ ]:
rows = []
names = SAMPLES + sorted({re.sub(r'^.*?\.(P56_sub\d+)\.tsv$', r'\1', p) for p in glob.glob(f'{LOG_DIR}/*.P56_sub*.tsv')},
                         key=lambda x: int(x.split('sub')[1]))
for s in names:
    r = {'name': s}
    r['cz_files'] = len(glob.glob(os.path.expanduser(f'~/Projects/mouse_dev/mapped/{s}/cz/*.cz'))) if s in SAMPLES else len(glob.glob(f'cz_subsets/{s}/*.cz'))
    r['mcds'] = sum(os.path.exists(f'mcds/{s}.{d}.mcds') for d in ('100kb', 'gene'))
    for p in ('allcools', 'cytozip'):
        r[f'{p}_h5ad'] = sum(os.path.exists(f'{p}/{s}/{s}.{st}.h5ad') for st in STAGES)
        r[f'{p}_log'] = os.path.exists(f'{LOG_DIR}/{p}.{s}.tsv')
    rows.append(r)
progress = pd.DataFrame(rows).set_index('name')
progress

## 2. Collect the per-stage logs

`logs/<pipeline>.<name>.tsv` — one row per stage written by `bench_run.py`. Stage labels are harmonised to
`mcds-100kb`, `mcds-gene` (pipeline A only) and `h5ad-100kb-CGN`, `h5ad-100kb-CHN`, `h5ad-gene-CGN`, `h5ad-gene-CHN`
(pipeline A writes `100kb-CGN` and `100kb-CHN` in a single `generate_region_adata` call → stage `h5ad-100kb-CGN+CHN`).

In [ ]:
logs = []
for f in sorted(glob.glob(f'{LOG_DIR}/*.tsv')):
    if f.endswith(('.size.tsv', '.cells.tsv')):   # size tables, not bench_run.py logs
        continue
    d = pd.read_csv(f, sep='\t')
    if 'pipeline' not in d.columns or 'wall_s' not in d.columns:
        continue
    d['log_file'] = os.path.basename(f)
    logs.append(d)
log = pd.concat(logs, ignore_index=True) if logs else pd.DataFrame()
if len(log):
    # keep the last successful run of every (pipeline, name, stage)
    log = log.sort_values('start_time').drop_duplicates(['pipeline', 'name', 'stage'], keep='last')
    log['exit_ok'] = log['exit_code'] == 0
    log['cpu_s'] = log['cpu_user_s'] + log['cpu_sys_s']
    log['wall_min'] = log['wall_s'] / 60
    log['max_pss_sum_gb'] = log['max_pss_sum_mb'] / 1024
    log['max_rss_sum_gb'] = log['max_rss_sum_mb'] / 1024
    log['is_subset'] = log['name'].str.contains('_sub')
    log.to_csv(f'{TABLE_DIR}/all_stages.tsv', sep='\t', index=False)
    bad = log.loc[~log.exit_ok, ['pipeline', 'name', 'stage', 'exit_code', 'log_file']]
    if len(bad):
        print('WARNING — failed stages (excluded below):'); display(bad)
    log = log[log.exit_ok]
    display(log.groupby(['pipeline', 'name']).size().unstack(0))
else:
    print('no logs yet — run the SLURM jobs first')
log.head()

## 3. Per-sample summary

For every (pipeline, sample): total wall time = Σ stages, total CPU time = Σ stages, peak memory = max over stages
(`max_pss_sum` is the fair physical footprint; `max_rss_sum` is also reported), output size = Σ h5ad sizes
(+ the intermediate `.mcds` for pipeline A). `cytozip (+allc2cz)` additionally charges the one-off `allc → cz`
conversion of that sample to pipeline B.

In [ ]:
def summarise(df):
    g = df.groupby(['pipeline', 'name'])
    s = pd.DataFrame({
        'n_cells': g['n_cells'].max(),
        'ncpu': g['ncpu'].max(),
        'n_stages': g.size(),
        'wall_s': g['wall_s'].sum(),
        'cpu_s': g['cpu_s'].sum(),
        'peak_pss_mb': g['max_pss_sum_mb'].max(),
        'peak_rss_mb': g['max_rss_sum_mb'].max(),
        'peak_rss_single_mb': g['max_rss_single_mb'].max(),
        'output_mb': g['output_size_mb'].sum(),
    }).reset_index()
    s['wall_min'] = s['wall_s'] / 60
    s['cpu_h'] = s['cpu_s'] / 3600
    s['peak_pss_gb'] = s['peak_pss_mb'] / 1024
    s['peak_rss_gb'] = s['peak_rss_mb'] / 1024
    return s

main = log[log.pipeline.isin(['allcools', 'cytozip'])]
summary = summarise(main)
# h5ad-only output size (exclude intermediate mcds) for a like-for-like file-size comparison
h5ad_only = main[main.stage.str.startswith('h5ad')].groupby(['pipeline', 'name'])['output_size_mb'].sum().rename('h5ad_mb').reset_index()
summary = summary.merge(h5ad_only, on=['pipeline', 'name'], how='left')

# cytozip + allc2cz (one-off conversion charged to pipeline B)
a2c = log[log.pipeline == 'allc2cz'].set_index('name')
if len(a2c):
    cz_plus = summary[summary.pipeline == 'cytozip'].copy()
    cz_plus['pipeline'] = 'cytozip+allc2cz'
    base = cz_plus['name'].str.replace(r'_sub\d+$', '', regex=True)  # subsets share the parent's conversion
    frac = np.where(cz_plus['name'].str.contains('_sub'), cz_plus['n_cells'] / base.map(a2c['n_cells']), 1.0)
    cz_plus['wall_s'] += base.map(a2c['wall_s']).values * frac
    cz_plus['cpu_s'] += base.map(a2c['cpu_s']).values * frac
    cz_plus['peak_pss_mb'] = np.maximum(cz_plus['peak_pss_mb'], base.map(a2c['max_pss_sum_mb']).values)
    cz_plus['peak_rss_mb'] = np.maximum(cz_plus['peak_rss_mb'], base.map(a2c['max_rss_sum_mb']).values)
    cz_plus['wall_min'] = cz_plus['wall_s'] / 60; cz_plus['cpu_h'] = cz_plus['cpu_s'] / 3600; cz_plus['peak_pss_gb'] = cz_plus['peak_pss_mb'] / 1024; cz_plus['peak_rss_gb'] = cz_plus['peak_rss_mb'] / 1024
    summary = pd.concat([summary, cz_plus], ignore_index=True)

summary['is_subset'] = summary['name'].str.contains('_sub')
summary.to_csv(f'{TABLE_DIR}/summary_per_sample.tsv', sep='\t', index=False)
full = summary[~summary.is_subset]
display(full.pivot(index='name', columns='pipeline', values=['n_cells', 'wall_min', 'cpu_h', 'peak_pss_gb', 'peak_rss_gb', 'h5ad_mb']).reindex(SAMPLES).round(2))

In [ ]:
# speed-up and memory ratio per sample (ALLCools / cytozip)
piv = full.pivot(index='name', columns='pipeline')
ratio = pd.DataFrame({
    'n_cells': piv['n_cells']['cytozip'],
    'speedup_wall (ALLCools/cytozip)': piv['wall_s']['allcools'] / piv['wall_s']['cytozip'],
    'speedup_wall (ALLCools/cytozip+allc2cz)': piv['wall_s']['allcools'] / piv['wall_s'].get('cytozip+allc2cz', np.nan),
    'speedup_cpu (ALLCools/cytozip)': piv['cpu_s']['allcools'] / piv['cpu_s']['cytozip'],
    'mem_ratio_pss (cytozip/ALLCools)': piv['peak_pss_mb']['cytozip'] / piv['peak_pss_mb']['allcools'],
    'mem_ratio_rss (cytozip/ALLCools)': piv['peak_rss_mb']['cytozip'] / piv['peak_rss_mb']['allcools'],
    'h5ad_size_ratio (cytozip/ALLCools)': piv['h5ad_mb']['cytozip'] / piv['h5ad_mb']['allcools'],
}).reindex(SAMPLES)
ratio.loc['mean'] = ratio.mean(numeric_only=True)
ratio.to_csv(f'{TABLE_DIR}/speedup_per_sample.tsv', sep='\t')
display(ratio.round(2))

tot = full.groupby('pipeline')[['wall_s', 'cpu_s']].sum()
print(textwrap.dedent(f'''
    === totals over {full.name.nunique()} samples, {int(full.groupby("pipeline").n_cells.sum().max()):,} cells ===
    ALLCools          wall = {tot.loc["allcools","wall_s"]/3600:6.2f} h   cpu = {tot.loc["allcools","cpu_s"]/3600:7.1f} CPU-h
    cytozip           wall = {tot.loc["cytozip","wall_s"]/3600:6.2f} h   cpu = {tot.loc["cytozip","cpu_s"]/3600:7.1f} CPU-h
    speed-up (wall)   = {tot.loc["allcools","wall_s"]/tot.loc["cytozip","wall_s"]:.2f}x
    speed-up (cpu)    = {tot.loc["allcools","cpu_s"]/tot.loc["cytozip","cpu_s"]:.2f}x
    peak PSS (max over samples)  ALLCools = {full[full.pipeline=="allcools"].peak_pss_gb.max():.1f} GB   cytozip = {full[full.pipeline=="cytozip"].peak_pss_gb.max():.1f} GB
'''))

## 4. Figure 1 — overall comparison (mean ± SD over the 10 age groups)

Panels: total wall time, total CPU time, peak memory (Σ PSS), h5ad output size.

In [ ]:
order = ['allcools', 'cytozip'] + (['cytozip+allc2cz'] if 'cytozip+allc2cz' in full.pipeline.unique() else [])
labels = {'allcools': 'ALLCools', 'cytozip': 'cytozip', 'cytozip+allc2cz': 'cytozip\n(+allc2cz)'}
cols = {'allcools': COLORS['ALLCools'], 'cytozip': COLORS['cytozip'], 'cytozip+allc2cz': COLORS['cytozip (+allc2cz)']}
metrics = [('wall_min', 'Wall time (min)', 'Runtime'),
           ('cpu_h', 'CPU time (CPU-h)', 'CPU time'),
           ('peak_pss_gb', 'Peak memory (GB, Σ PSS)', 'Peak memory'),
           ('h5ad_mb', 'h5ad size (MB)', 'Output size')]

fig, axes = plt.subplots(1, 4, figsize=(7.2, 2.4), constrained_layout=True)
for ax, (col, ylabel, title) in zip(axes, metrics):
    d = full.groupby('pipeline')[col].agg(['mean', 'std']).reindex(order)
    x = np.arange(len(order))
    ax.bar(x, d['mean'], yerr=d['std'], capsize=3, width=0.65, color=[cols[p] for p in order],
           error_kw=dict(ecolor='black', lw=0.8))
    # individual samples as dots
    for i, p in enumerate(order):
        v = full.loc[full.pipeline == p, col].values
        ax.scatter(np.random.default_rng(0).uniform(-0.15, 0.15, len(v)) + i, v, s=6, color='k', alpha=0.6, zorder=3, lw=0)
    ax.set_xticks(x); ax.set_xticklabels([labels[p] for p in order], fontsize=7)
    ax.set_ylabel(ylabel); ax.set_title(title, fontsize=9); despine(ax)
    if col in ('wall_min', 'cpu_h'):
        sp = d.loc['allcools', 'mean'] / d.loc['cytozip', 'mean']
        ax.text(0.98, 0.98, f'{sp:.1f}× faster', transform=ax.transAxes, ha='right', va='top', fontsize=7, color=COLORS['cytozip'])
    elif col == 'peak_pss_gb':
        r = d.loc['cytozip', 'mean'] / d.loc['allcools', 'mean']
        ax.text(0.98, 0.98, f'{r*100:.0f}% memory', transform=ax.transAxes, ha='right', va='top', fontsize=7, color=COLORS['cytozip'])
fig.suptitle(f'allc → mcds → h5ad (ALLCools) vs cz → h5ad (cytozip); n = {full.name.nunique()} samples, mean ± SD', fontsize=8)
savefig(fig, 'fig1_overall_mean_sd')
plt.show()

## 5. Figure 2 — stage breakdown (stacked wall time per pipeline)

In [ ]:
stage_order_a = ['mcds-100kb', 'mcds-gene', 'h5ad-100kb-CGN+CHN', 'h5ad-gene-CGN', 'h5ad-gene-CHN']
stage_order_b = ['h5ad-100kb-CGN', 'h5ad-100kb-CHN', 'h5ad-gene-CGN', 'h5ad-gene-CHN']
stage_mean = main[~main.is_subset].groupby(['pipeline', 'stage'])['wall_min'].mean().unstack(0)
stage_mean.to_csv(f'{TABLE_DIR}/stage_mean_wall_min.tsv', sep='\t')
display(stage_mean.round(2))

fig, ax = plt.subplots(figsize=(3.2, 2.6), constrained_layout=True)
cmap_a = plt.get_cmap('Blues')(np.linspace(0.45, 0.9, len(stage_order_a)))
cmap_b = plt.get_cmap('Reds')(np.linspace(0.45, 0.9, len(stage_order_b)))
for xi, (p, stages, cm) in enumerate([('allcools', stage_order_a, cmap_a), ('cytozip', stage_order_b, cmap_b)]):
    bottom = 0
    for st, c in zip(stages, cm):
        v = stage_mean[p].get(st, np.nan) if p in stage_mean else np.nan
        if np.isnan(v):
            continue
        ax.bar(xi, v, bottom=bottom, color=c, width=0.7, edgecolor='white', lw=0.4, label=f'{LABEL[p]}: {st}')
        bottom += v
    ax.text(xi, bottom, f'{bottom:.1f} min', ha='center', va='bottom', fontsize=7)
ax.set_xticks([0, 1]); ax.set_xticklabels(['ALLCools', 'cytozip'], fontsize=7)
ax.set_xlim(-0.6, 1.6)
ax.set_title('allc→mcds→h5ad  vs  cz→h5ad', fontsize=8)
ax.set_ylabel('Mean wall time per sample (min)'); despine(ax)
ax.legend(fontsize=6, frameon=False, bbox_to_anchor=(1.02, 1), loc='upper left', title='stage', title_fontsize=6.5)
savefig(fig, 'fig2_stage_breakdown')
plt.show()

## 6. Figure 3 — per-sample runtime and peak memory

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.5), constrained_layout=True)
x = np.arange(len(SAMPLES)); w = 0.38
for ax, (col, ylabel) in zip(axes, [('wall_min', 'Wall time (min)'), ('peak_pss_gb', 'Peak memory (GB, Σ PSS)')]):
    for i, p in enumerate(['allcools', 'cytozip']):
        v = full[full.pipeline == p].set_index('name')[col].reindex(SAMPLES)
        ax.bar(x + (i - 0.5) * w, v, width=w, color=cols[p], label=LABEL[p])
    ax.set_xticks(x); ax.set_xticklabels(SAMPLES, rotation=45, ha='right', fontsize=7)
    ax.set_ylabel(ylabel); despine(ax)
    ax2 = ax.twinx(); ax2.plot(x, full[full.pipeline == 'cytozip'].set_index('name')['n_cells'].reindex(SAMPLES) / 1000, 'k.--', lw=0.6, ms=3)
    ax2.set_ylabel('cells (×1000)', fontsize=7); ax2.spines['top'].set_visible(False)
axes[0].legend(frameon=False, fontsize=7)
savefig(fig, 'fig3_per_sample')
plt.show()

## 7. Figure 4 — scaling with the number of cells

Nested random subsets of P56 (500 … 16 000 cells) plus all full samples; log–log axes with a fitted power-law slope.

In [ ]:
sc = summary[summary.pipeline.isin(['allcools', 'cytozip'])].copy()
if sc.is_subset.any():
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4), constrained_layout=True)
    for ax, (col, ylabel) in zip(axes, [('wall_min', 'Wall time (min)'), ('cpu_h', 'CPU time (CPU-h)'), ('peak_pss_gb', 'Peak memory (GB, Σ PSS)')]):
        for p in ['allcools', 'cytozip']:
            d = sc[sc.pipeline == p].sort_values('n_cells')
            sub, fl = d[d.is_subset], d[~d.is_subset]
            ax.plot(sub.n_cells, sub[col], '-o', color=cols[p], ms=3.5, lw=1, label=f'{LABEL[p]} (P56 subsets)')
            ax.scatter(fl.n_cells, fl[col], marker='s', s=14, facecolor='none', edgecolor=cols[p], lw=0.8, label=f'{LABEL[p]} (full samples)')
            if len(sub) >= 3:
                k, b = np.polyfit(np.log10(sub.n_cells), np.log10(sub[col].clip(lower=1e-6)), 1)
                xs = np.array([sub.n_cells.min(), sub.n_cells.max()])
                ax.plot(xs, 10 ** (k * np.log10(xs) + b), ':', color=cols[p], lw=0.8)
                ax.text(0.02, 0.98 - 0.09 * (p == 'cytozip'), f'{LABEL[p]}: slope {k:.2f}', transform=ax.transAxes, fontsize=6.5, color=cols[p], va='top')
        ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('Number of cells'); ax.set_ylabel(ylabel); despine(ax)
    axes[-1].legend(fontsize=6, frameon=False, loc='lower right')
    savefig(fig, 'fig4_scaling_vs_cells')
    plt.show()
else:
    print('no scaling-series results yet (run scripts/04_make_subsets.sh then scripts/05_scaling.sh)')

## 8. `allc → cz` conversion: runtime, memory and file sizes

`scripts/01_allc2cz.sh <age>` runs `czip allc2cz -I allc/ -O cz/ -r mm10_with_chrL.allc.cz -j A2C_NCPU` (batch mode: the
reference is decoded once and shared with all workers by fork) under `bench_run.py`, so for every age group we have

* `logs/allc2cz.<age>.tsv` — wall / CPU time, peak Σ RSS / Σ PSS, largest single-process RSS, number of CPUs
* `logs/allc2cz.<age>.size.tsv` — total size of all `allc.tsv.gz` vs all `.cz`
* `logs/allc2cz.<age>.cells.tsv` — per-cell sizes (`allc.tsv.gz`, `.tbi`, `.cz`)


In [ ]:
size_files = glob.glob(f'{LOG_DIR}/allc2cz.*.size.tsv')
if size_files and len(a2c):
    sz = pd.concat([pd.read_csv(f, sep='\t') for f in size_files]).set_index('name').reindex(SAMPLES)
    sz['allc_gb'] = sz.allc_dir_mb / 1024; sz['cz_gb'] = sz.cz_dir_mb / 1024
    sz['cz/allc'] = sz.cz_dir_mb / sz.allc_dir_mb
    sz['allc_mb_per_cell'] = sz.allc_dir_mb / sz.n_allc; sz['cz_mb_per_cell'] = sz.cz_dir_mb / sz.n_cz
    sz['ncpu'] = a2c['ncpu'].reindex(SAMPLES)
    sz['wall_min'] = a2c['wall_s'].reindex(SAMPLES) / 60
    sz['cpu_h'] = a2c['cpu_s'].reindex(SAMPLES) / 3600
    sz['s_per_cell_wall'] = a2c['wall_s'].reindex(SAMPLES) / sz.n_allc
    sz['cpu_s_per_cell'] = a2c['cpu_s'].reindex(SAMPLES) / sz.n_allc
    sz['peak_pss_gb'] = a2c['max_pss_sum_mb'].reindex(SAMPLES) / 1024
    sz['peak_rss_gb'] = a2c['max_rss_sum_mb'].reindex(SAMPLES) / 1024
    sz['peak_rss_single_gb'] = a2c['max_rss_single_mb'].reindex(SAMPLES) / 1024
    sz['max_n_procs'] = a2c['max_n_procs'].reindex(SAMPLES)
    sz.loc['total/mean'] = sz.sum(numeric_only=True)
    for c in ['cz/allc', 'allc_mb_per_cell', 'cz_mb_per_cell', 'ncpu', 's_per_cell_wall', 'cpu_s_per_cell', 'peak_pss_gb', 'peak_rss_gb', 'peak_rss_single_gb', 'max_n_procs']:
        sz.loc['total/mean', c] = sz.loc[SAMPLES, c].mean() if c != 'cz/allc' else sz.loc[SAMPLES, 'cz_dir_mb'].sum() / sz.loc[SAMPLES, 'allc_dir_mb'].sum()
    sz.to_csv(f'{TABLE_DIR}/allc2cz_summary.tsv', sep='\t')
    display(sz[['n_allc', 'n_cz', 'ncpu', 'wall_min', 'cpu_h', 's_per_cell_wall', 'cpu_s_per_cell', 'peak_pss_gb', 'peak_rss_gb',
                'peak_rss_single_gb', 'allc_gb', 'cz_gb', 'cz/allc', 'allc_mb_per_cell', 'cz_mb_per_cell']].round(3))
    t = sz.loc['total/mean']
    print(f"allc2cz over {int(t.n_allc):,} cells: wall = {t.wall_min/60:.2f} h, cpu = {t.cpu_h:.1f} CPU-h, "
          f"mean peak PSS = {t.peak_pss_gb:.1f} GB; storage {t.allc_gb:.1f} GB allc -> {t.cz_gb:.1f} GB cz ({t['cz/allc']*100:.1f}%)")

    # per-cell sizes
    cell_files = glob.glob(f'{LOG_DIR}/allc2cz.*.cells.tsv')
    pc = pd.concat([pd.read_csv(f, sep='\t').assign(name=os.path.basename(f).split('.')[1]) for f in cell_files], ignore_index=True)
    pc = pc[pc.cz_bytes.notna()].copy()
    pc['allc_mb'] = pc.allc_bytes / 1024**2; pc['cz_mb'] = pc.cz_bytes.astype(float) / 1024**2; pc['ratio'] = pc.cz_bytes.astype(float) / pc.allc_bytes
    pc.to_csv(f'{TABLE_DIR}/allc2cz_per_cell_sizes.tsv.gz', sep='\t', index=False)
    print(f"per cell (n={len(pc):,}): allc median {pc.allc_mb.median():.1f} MB, cz median {pc.cz_mb.median():.1f} MB, "
          f"median cz/allc = {pc.ratio.median():.3f}")

    fig, axes = plt.subplots(1, 4, figsize=(7.4, 2.3), constrained_layout=True)
    x = np.arange(len(SAMPLES)); w = 0.38; s_ = sz.loc[SAMPLES]
    ax = axes[0]
    ax.bar(x, s_.wall_min, color=COLORS['cytozip'], width=0.65)
    ax.set_ylabel('Wall time (min)'); ax.set_title(f'allc→cz ({int(s_.ncpu.max())} CPUs)', fontsize=8)
    ax2 = ax.twinx(); ax2.plot(x, s_.n_allc / 1000, 'k.--', lw=0.6, ms=3); ax2.set_ylabel('cells (×1000)', fontsize=7); ax2.spines['top'].set_visible(False)
    ax = axes[1]
    ax.bar(x - w/2, s_.peak_pss_gb, w, color=COLORS['cytozip'], label='Σ PSS')
    ax.bar(x + w/2, s_.peak_rss_gb, w, color='#f4a261', label='Σ RSS')
    ax.set_ylabel('Peak memory (GB)'); ax.set_title('Peak memory', fontsize=8)
    ax.set_ylim(0, max(s_.peak_rss_gb.max(), s_.peak_pss_gb.max()) * 1.3); ax.legend(frameon=False, fontsize=6, loc='upper left', ncol=2)
    ax = axes[2]
    ax.bar(x - w/2, s_.allc_gb, w, color='#7f7f7f', label='allc.tsv.gz')
    ax.bar(x + w/2, s_.cz_gb, w, color=COLORS['cytozip'], label='.cz')
    ax.set_ylabel('Total size (GB)'); ax.set_title('Storage', fontsize=8)
    ax.set_ylim(0, s_.allc_gb.max() * 1.3); ax.legend(frameon=False, fontsize=6, loc='upper left')
    for ax in axes[:3]:
        ax.set_xticks(x); ax.set_xticklabels(SAMPLES, rotation=45, ha='right', fontsize=6.5); despine(ax)
    ax = axes[3]
    ax.hexbin(pc.allc_mb, pc.cz_mb, gridsize=50, bins='log', cmap='Reds', lw=0, mincnt=1)
    lim = max(pc.allc_mb.max(), pc.cz_mb.max())
    for r_ in (0.1, 0.2, 0.5):
        ax.plot([0, lim], [0, lim * r_], 'k:', lw=0.5); ax.text(lim, lim * r_, f'{r_:g}×', fontsize=5.5, va='bottom', ha='right')
    ax.set_xlabel('allc.tsv.gz size per cell (MB)'); ax.set_ylabel('.cz size per cell (MB)')
    ax.set_title(f'per cell (n={len(pc):,}), median cz/allc = {pc.ratio.median():.2f}', fontsize=7); despine(ax)
    savefig(fig, 'fig5_allc2cz_time_memory_storage')
    plt.show()
else:
    print('no allc2cz logs yet')

## 9. Validation — do both pipelines produce the same matrices?

For one sample, load the `100kb-CGN` / `gene-CGN` (and CHN) h5ad from both pipelines, align cells and features and compare
the mC fractions in `.X` on entries covered in both. Expected: Pearson r ≈ 1 with tiny residual differences coming from

* blacklist handling — pym3c removes bins overlapping the blacklist by ≥ 20 % of their length (`f=0.2`),
  cytozip drops any feature overlapping a blacklist interval;
* `cov` capping in `.cz` (`uint8`, mc scaled with cov so the fraction is preserved);
* chrX/chrY/chrM gene exclusion differences for the gene matrices.

In [ ]:
import anndata as ad
import scipy.sparse as sp

VAL_SAMPLE = 'P0'   # any finished sample

def _var_key(adata, var_dim):
    v = adata.var
    cand = [(f'{var_dim}_chrom', f'{var_dim}_start', f'{var_dim}_end'), ('chrom', 'start', 'end')]
    for c, s, e in cand:
        if c in v and s in v and e in v:
            return pd.Index(v[c].astype(str) + ':' + v[s].astype(int).astype(str) + '-' + v[e].astype(int).astype(str))
    return adata.var_names  # gene ids

def compare(sample, feat, mc):
    fa = f'allcools/{sample}/{sample}.{feat}-{mc}.h5ad'; fb = f'cytozip/{sample}/{sample}.{feat}-{mc}.h5ad'
    if not (os.path.exists(fa) and os.path.exists(fb)):
        print('missing', fa, fb); return None
    A, B = ad.read_h5ad(fa), ad.read_h5ad(fb)
    A.var_names = _var_key(A, feat); B.var_names = _var_key(B, feat)
    cells = A.obs_names.intersection(B.obs_names); feats = A.var_names.intersection(B.var_names)
    A, B = A[cells, feats], B[cells, feats]
    xa = A.X.toarray() if sp.issparse(A.X) else np.asarray(A.X)
    xb = B.X.toarray() if sp.issparse(B.X) else np.asarray(B.X)
    xa, xb = xa.astype(np.float64), xb.astype(np.float64)
    if 'cov' in B.layers:   # mask to entries covered in cytozip (uncovered = NaN / prior)
        cov = B.layers['cov'].toarray() if sp.issparse(B.layers['cov']) else np.asarray(B.layers['cov'])
        m = (cov > 0) & np.isfinite(xa) & np.isfinite(xb)
    else:
        m = np.isfinite(xa) & np.isfinite(xb)
    r = np.corrcoef(xa[m], xb[m])[0, 1]
    diff = np.abs(xa[m] - xb[m])
    res = dict(sample=sample, feature=feat, mc=mc, n_cells_A=A.n_obs, n_cells_B=B.n_obs, n_cells_shared=len(cells),
               n_feat_A=A.n_vars, n_feat_B=B.n_vars, n_feat_shared=len(feats), n_entries=int(m.sum()),
               pearson_r=r, max_abs_diff=diff.max(), median_abs_diff=np.median(diff), frac_diff_gt_0p01=(diff > 0.01).mean())
    return res, xa[m], xb[m]

val_rows, pairs = [], {}
for feat in ['100kb', 'gene']:
    for mc in ['CGN', 'CHN']:
        out = compare(VAL_SAMPLE, feat, mc)
        if out:
            val_rows.append(out[0]); pairs[(feat, mc)] = out[1:]
val = pd.DataFrame(val_rows)
val.to_csv(f'{TABLE_DIR}/validation_{VAL_SAMPLE}.tsv', sep='\t', index=False)
display(val)

if pairs:
    fig, axes = plt.subplots(1, len(pairs), figsize=(1.9 * len(pairs), 2.0), constrained_layout=True)
    for ax, ((feat, mc), (xa, xb)) in zip(np.atleast_1d(axes), pairs.items()):
        rng = np.random.default_rng(0); idx = rng.choice(len(xa), min(200_000, len(xa)), replace=False)
        ax.hexbin(xa[idx], xb[idx], gridsize=60, bins='log', cmap='viridis', lw=0)
        ax.plot([0, 1], [0, 1], 'r--', lw=0.6)
        ax.set_title(f'{feat} m{mc[:2]}  r = {np.corrcoef(xa, xb)[0,1]:.4f}', fontsize=7)
        ax.set_xlabel('ALLCools / pym3c'); ax.set_ylabel('cytozip'); despine(ax)
    savefig(fig, f'fig6_validation_{VAL_SAMPLE}')
    plt.show()

## 10. Numbers for the manuscript text

In [ ]:
if len(full):
    g = full.groupby('pipeline')
    ncell = int(full[full.pipeline == 'cytozip'].n_cells.sum())
    txt = f'''
Across {full.name.nunique()} samples ({ncell:,} cells; {int(full.ncpu.max())} CPUs per job),
building the four feature matrices (100-kb bins and gene bodies, mCG and mCH) took on average
{g.wall_min.mean()['allcools']:.1f} ± {g.wall_min.std()['allcools']:.1f} min per sample with the allc → mcds → h5ad route (ALLCools + pym3c)
versus {g.wall_min.mean()['cytozip']:.1f} ± {g.wall_min.std()['cytozip']:.1f} min with cz → h5ad (cytozip), a
{g.wall_s.sum()['allcools'] / g.wall_s.sum()['cytozip']:.1f}-fold reduction in wall time
({g.cpu_s.sum()['allcools'] / g.cpu_s.sum()['cytozip']:.1f}-fold in CPU time).
Peak memory (proportional set size summed over all worker processes) dropped from
{g.peak_pss_gb.mean()['allcools']:.1f} ± {g.peak_pss_gb.std()['allcools']:.1f} GB to
{g.peak_pss_gb.mean()['cytozip']:.1f} ± {g.peak_pss_gb.std()['cytozip']:.1f} GB
({g.peak_pss_gb.mean()['cytozip'] / g.peak_pss_gb.mean()['allcools'] * 100:.0f} %).'''
    if 'cytozip+allc2cz' in g.groups:
        txt += f'''
Including the one-off allc → cz conversion, cytozip still needed only
{g.wall_min.mean()['cytozip+allc2cz']:.1f} ± {g.wall_min.std()['cytozip+allc2cz']:.1f} min per sample
({g.wall_s.sum()['allcools'] / g.wall_s.sum()['cytozip+allc2cz']:.1f}-fold faster than ALLCools).'''
    print(txt)
    open(f'{TABLE_DIR}/manuscript_numbers.txt', 'w').write(txt)